# 03 – The final exam: evaluating the model

In notebook 02 the model trained, and we checked it on the **validation** set after every epoch. But we also *chose* the best epoch using that validation set, so its score is slightly flattering, like grading yourself on practice tests you've seen.

Now we use the **test set**: 2,700 images that played no part in training *or* in choosing the model. This is the number that goes in your paper.

**In this notebook:**
1. Rebuild the exact same test set as notebook 02
2. Load your trained model
3. Overall test accuracy
4. Per-class scores: precision, recall, F1
5. The **confusion matrix**: which classes get mixed up
6. Its most confident mistakes
7. Is it confident when it's wrong? (your answer to question 4 in notebook 02, tested properly)

Runs in about a minute. No training happens here. **Run → Run All Cells**, then read along.

## Step 1 – Setup

Same libraries as before, plus two new ones:
- `sklearn.metrics` (**scikit-learn**) has ready-made functions for the standard scores and the confusion matrix
- `seaborn` draws nicer heatmaps than plain matplotlib

Figures for your paper get saved into a `figures` folder.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from tqdm.auto import tqdm

DATA_DIR = Path("C:/data/EuroSAT")
MODEL_PATH = Path("../models/resnet50_eurosat.pth")
FIG_DIR = Path("../figures")
FIG_DIR.mkdir(exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)
print("Model file found:", MODEL_PATH.exists())

## Step 2 – Rebuild the exact same test set

This must be **exactly** the 2,700 images that were set aside in notebook 02. If even a few training images sneaked in, the score would be dishonest.

That's why notebook 02 shuffled with `manual_seed(42)`: the same seed produces the same "random" order every time. We repeat those exact steps and take the last 10%.

We use the evaluation transform: resize + normalise, **no flips**. We're testing on real, unmodified images.

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

eval_data = datasets.ImageFolder(DATA_DIR, transform=eval_transform)
class_names = eval_data.classes

shuffled = torch.randperm(len(eval_data), generator=torch.Generator().manual_seed(42)).tolist()
n_train = int(0.8 * len(shuffled))
n_val   = int(0.1 * len(shuffled))
test_set = Subset(eval_data, shuffled[n_train + n_val:])

test_loader = DataLoader(test_set, batch_size=64, shuffle=False, num_workers=8, pin_memory=True)
print(f"Test images: {len(test_set)}")

## Step 3 – Load your trained model

A `.pth` file holds only the **learned numbers** (the weights), not the network's structure. So we first build an empty ResNet50 with the same shape (same 10-class last layer), then pour the saved numbers into it.

`weights=None` = don't download ImageNet weights this time, since we're about to replace them all with yours anyway.

`model.eval()` puts it in testing mode.

In [ ]:
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device, memory_format=torch.channels_last).eval()
print("Model loaded.")

## Step 4 – Run the model on every test image

For each test image we keep three things:
- the **true** label
- the **predicted** label (highest score)
- the **confidence**: how sure it was, from `softmax` (turns 10 scores into probabilities adding up to 100%)

We also keep all 10 probabilities per image, in case we want them later.

In [ ]:
all_labels, all_preds, all_probs = [], [], []

with torch.no_grad(), torch.autocast(device, dtype=torch.bfloat16):
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(device, memory_format=torch.channels_last)
        probs = torch.softmax(model(images).float(), dim=1).cpu()
        all_probs.append(probs)
        all_preds.append(probs.argmax(1))
        all_labels.append(labels)

all_probs  = torch.cat(all_probs).numpy()
all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()
confidence = all_probs.max(axis=1)

print("Shape of all probabilities:", all_probs.shape, " ->  (images, classes)")

## Step 5 – Overall accuracy

The headline number: what fraction of test images did it get right?

In [ ]:
test_accuracy = accuracy_score(all_labels, all_preds)
n_wrong = (all_preds != all_labels).sum()
print(f"Test accuracy: {test_accuracy:.2%}")
print(f"Wrong: {n_wrong} out of {len(all_labels)} images")

Compare this with the best validation accuracy from notebook 02. They should be close. If test were much lower, it would mean we'd accidentally tuned the model to the validation set.

## Step 6 – Per-class scores: precision, recall, F1

One overall number hides a lot. A model could be perfect on Forest and terrible on River and still look good overall. So we score every class separately.

Take **Forest** as the example:

| Score | Question it answers | Low score means… |
|---|---|---|
| **Precision** | Of all the images the model *called* Forest, how many really were Forest? | It cries "Forest!" too often (false alarms) |
| **Recall** | Of all the images that *really were* Forest, how many did it find? | It misses forests |
| **F1-score** | One number combining both (their "harmonic mean") | Either problem |
| **Support** | How many test images of this class there were | (just a count) |

**Why both matter for deforestation:**
- Low **Forest recall** → it misses real forest, so it may report deforestation that didn't happen (a false alarm)
- Low **Forest precision** → it calls other things forest, so it may miss real deforestation

At the bottom: **macro avg** = simple average over the 10 classes (each class counts equally); **weighted avg** = average weighted by how many images each class has.

In [ ]:
print(classification_report(all_labels, all_preds, target_names=class_names, digits=3))

The same per-class scores as a chart, sorted from best to worst, so the weak classes jump out.

In [ ]:
report = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True)
f1 = {name: report[name]["f1-score"] for name in class_names}
order = sorted(f1, key=f1.get)

plt.figure(figsize=(9, 4.5))
bars = plt.barh(order, [f1[n] * 100 for n in order], color="seagreen")
plt.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=9)
plt.xlim(min(f1.values()) * 100 - 5, 101)
plt.xlabel("F1-score (%)")
plt.title("Test F1-score per class (worst at the top)")
plt.tight_layout()
plt.savefig(FIG_DIR / "f1_per_class.png", dpi=150)
plt.show()

## Step 7 – The confusion matrix

The most important figure in this notebook. It's a 10 × 10 table:
- each **row** is the **true** class
- each **column** is what the model **predicted**
- the number in a cell = how many images of *that row's class* were predicted as *that column's class*

So the **diagonal** (top-left to bottom-right) is the correct answers. **Everything off the diagonal is a mistake**, and it tells you exactly what got confused with what.

Example: if the cell at row *River*, column *Highway* says 7, then 7 real river images were called highways.

We draw it twice:
1. **Counts**, the raw numbers
2. **Percentages of each row**, which makes classes with different sizes comparable (each row adds up to 100%)

In [ ]:
short = ["Annual\nCrop", "Forest", "Herb.\nVeg", "Highway", "Indus-\ntrial", "Pasture", "Perm.\nCrop", "Resid-\nential", "River", "Sea/\nLake"]
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=short, yticklabels=short, cbar=False)
plt.xlabel("Predicted class", fontsize=12)
plt.ylabel("True class", fontsize=12)
plt.title(f"Confusion matrix – test set ({len(all_labels)} images, accuracy {test_accuracy:.1%})", fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / "confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
cm_percent = cm / cm.sum(axis=1, keepdims=True) * 100

plt.figure(figsize=(10, 8))
sns.heatmap(cm_percent, annot=True, fmt=".1f", cmap="Blues", xticklabels=short, yticklabels=short,
            cbar_kws={"label": "% of the true class"})
plt.xlabel("Predicted class", fontsize=12)
plt.ylabel("True class", fontsize=12)
plt.title("Confusion matrix – % of each true class", fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / "confusion_matrix_percent.png", dpi=150)
plt.show()

### The biggest mix-ups, as a list

Reading a 10 × 10 grid is hard, so let's pull out the off-diagonal cells with the most mistakes.

`np.fill_diagonal(..., 0)` blanks out the correct answers so only the mistakes are left; then we sort them.

In [ ]:
mistakes = cm.copy()
np.fill_diagonal(mistakes, 0)

pairs = [(mistakes[t, p], class_names[t], class_names[p]) for t in range(10) for p in range(10) if mistakes[t, p] > 0]
pairs.sort(reverse=True)

print("Most common mistakes (true → predicted):\n")
for count, true_name, pred_name in pairs[:8]:
    print(f"  {count:3d} × {true_name:22} → {pred_name}")

Do these match what you predicted in notebook 01 when you looked at the image grid?

## Step 8 – The most confident mistakes

These are the most interesting errors: the model was **sure** and still **wrong**. Sorted by confidence, highest first.

Look closely. Some may be genuinely ambiguous (a field with a road through it: is that "Highway" or "Annual Crop"?). Some might even be **labelling errors in the dataset** itself. Real datasets have those.

In [ ]:
def to_picture(tensor):
    t = tensor * torch.tensor(STD).view(3, 1, 1) + torch.tensor(MEAN).view(3, 1, 1)
    return t.clamp(0, 1).permute(1, 2, 0).numpy()

wrong = np.where(all_preds != all_labels)[0]
wrong = wrong[np.argsort(-confidence[wrong])]   # most confident first

fig, axes = plt.subplots(2, 8, figsize=(18, 5.5), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for ax, i in zip(axes.flat, wrong[:16]):
    image, _ = test_set[i]
    ax.imshow(to_picture(image))
    ax.set_title(f"said {class_names[all_preds[i]]} {confidence[i]:.0%}\ntrue: {class_names[all_labels[i]]}",
                 fontsize=8, color="red")
plt.suptitle("Most confident mistakes on the test set")
plt.tight_layout()
plt.savefig(FIG_DIR / "confident_mistakes.png", dpi=150)
plt.show()

## Step 9 – Is it confident when it's wrong?

In notebook 02 you said it's *usually confident* even when wrong. Let's test that properly with **all** the test images.

The chart compares how confident the model was on its **correct** answers (green) versus its **wrong** answers (red).

- If the red bars sit far to the **left** (low confidence), mistakes are easy to catch: just distrust low-confidence predictions.
- If the red bars spread to the **right** (high confidence), some mistakes are impossible to spot from confidence alone.

In [ ]:
correct = all_preds == all_labels

plt.figure(figsize=(10, 4.5))
bins = np.linspace(0.1, 1, 37)
plt.hist(confidence[correct], bins=bins, color="seagreen", alpha=0.7, label=f"Correct ({correct.sum()})")
plt.hist(confidence[~correct], bins=bins, color="crimson", alpha=0.8, label=f"Wrong ({(~correct).sum()})")
plt.yscale("log")   # log scale, otherwise the few wrong ones are invisible next to thousands of right ones
plt.xlabel("Model confidence")
plt.ylabel("Number of images (log scale)")
plt.title("How confident was the model?")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "confidence_correct_vs_wrong.png", dpi=150)
plt.show()

print(f"Average confidence when correct: {confidence[correct].mean():.1%}")
print(f"Average confidence when wrong:   {confidence[~correct].mean():.1%}")
print()
for threshold in (0.5, 0.7, 0.9):
    kept = confidence >= threshold
    print(f"If we only trust predictions ≥{threshold:.0%} confident: keep {kept.mean():.1%} of images, "
          f"accuracy on those = {correct[kept].mean():.2%}, mistakes left = {(~correct & kept).sum()}")

**Reading the threshold lines:** raising the confidence bar throws away some predictions but makes the ones you keep more accurate. This is a real tool for your deforestation maps: you could only report a change when the model is, say, ≥90% confident on *both* dates, and accept that you'll miss a few.

## Step 10 – Summary for your paper

In [ ]:
print("=== Results summary ===")
print(f"Test images:           {len(all_labels)}")
print(f"Test accuracy:         {test_accuracy:.2%}")
print(f"Macro-average F1:      {report['macro avg']['f1-score']:.2%}")
print(f"Best class:            {order[-1]} (F1 {f1[order[-1]]:.1%})")
print(f"Worst class:           {order[0]} (F1 {f1[order[0]]:.1%})")
print(f"Most common mistake:   {pairs[0][1]} → {pairs[0][2]} ({pairs[0][0]} images)")
print(f"Forest precision / recall: {report['Forest']['precision']:.1%} / {report['Forest']['recall']:.1%}")
print()
print("Figures saved in:", FIG_DIR.resolve())
for f in sorted(FIG_DIR.glob("*.png")):
    print("  ", f.name)

## Done!

You now have the **classifier half** of the paper: a test accuracy, per-class scores, a confusion matrix, and an honest look at where and how confidently it fails. The figures in `figures/` are small, so they *will* go to GitHub (unlike the model file).

**Next (notebook 04):** the actual research. Download real Sentinel-2 satellite images of a forest region from two different years, chop them into 64 × 64 patches, classify every patch, and map where forest turned into something else.

## Think about it (answer in chat)

1. Forest's precision and recall: which one matters more for *deforestation detection*, and why?
2. Pick the most common mistake from Step 7. Look at the confident-mistake images: why do you think the model confuses those two classes?
3. From Step 9: if you only trusted predictions ≥90% confident, how many mistakes would still slip through? Is that acceptable for a deforestation map?
4. EuroSAT images all come from **Europe**. Your study area will probably be tropical (Amazon, Southeast Asia, Africa). What might go wrong when this model sees a tropical rainforest for the first time?